In [1]:
# COLAB SETUP

%cd /content
!rm -rf /content/proto-tsrl
!git clone https://github.com/haiyan-wang/proto-tsrl.git /content/proto-tsrl
%cd /content/proto-tsrl

from google.colab import drive, runtime
drive.mount('/content/drive')

import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

/content
Cloning into '/content/proto-tsrl'...
remote: Enumerating objects: 374, done.
remote: Counting objects: 100% (374/374), done.
remote: Compressing objects: 100% (301/301), done.
remote: Total 374 (delta 198), reused 168 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (374/374), 3.75 MiB | 9.26 MiB/s, done.
Resolving deltas: 100% (198/198), done.
/content/proto-tsrl
Mounted at /content/drive
/content/proto-tsrl


In [ ]:
import functools

import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

from src.models.univariate_model import UnivariateModel
from src.utils.sampling_utils import *
from src.utils.training_utils import *

In [3]:
### SETTINGS

SEED = 42

# device
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

# dataloaders
BATCH_SIZE = 128
VAL_RATIO = 0.05
N_WORKERS = 4

# contrastive sampling
COLLATE_FN_KWARGS = {
    'min_len': 300,
    'max_len': 600,
    'num_neg': 10,
    'min_overlap': 0.5,
    'min_var': 5e-3,
    'max_var': 1e-2
}

# schedule
N_EPOCHS_STAGE1 = 30
N_EPOCHS_STAGE2 = 15
N_EPOCHS_STAGE3 = 10
TOTAL_EPOCHS = N_EPOCHS_STAGE1 + N_EPOCHS_STAGE2 + N_EPOCHS_STAGE3

# logging
CHECKPOINT_EPOCHS = [*range(1, TOTAL_EPOCHS + 1)] # 1-indexed
SAVE_DIR = "/content/drive/MyDrive/Duke/Senior Year/Thesis/experiments/autoregressive/checkpoints"

# architecture
MASK_PROB = 0.2
MID_WEIGHT = 0.5
PROTO_NEG_MARGIN = 0.1
PROTO_DIVERSITY_THRESHOLD = 0.2
LAMBDA_PROTO = 1.0
TEMPERATURE = 1.0
LAMBDA_REPR = 1e-3
GRADIENT_CLIP = 1.0

# parameter settings and optimizers
REP_DIMS = [300]
MODELS = []
OPTIMIZER_DICTS = []
CKPT_PATHS = []
for dim in REP_DIMS:
    model = UnivariateModel(representation_dimension = dim, mask_probability = MASK_PROB)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(DEVICE)
    MODELS.append(model)

    ckpt_path = f"{SAVE_DIR}/dim{dim}"
    CKPT_PATHS.append(ckpt_path)

    opt_dict = {
    "stage1": torch.optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-5),
    "stage2": torch.optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-5),
    "stage3": torch.optim.Adam(model.parameters(), lr = 1e-4, weight_decay = 1e-5),
    }
    OPTIMIZER_DICTS.append(opt_dict)

SCHEDULER = None

cuda:0


In [ ]:
### LOAD DATA

with np.load("/content/drive/MyDrive/Duke/Senior Year/Thesis/data/autoregressive.npz", allow_pickle = True) as data:
    series = list(data["series"])
y = np.loadtxt("/content/drive/MyDrive/Duke/Senior Year/Thesis/data/autoregressive_labels.csv", delimiter = ',')

# shuffle
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(series))
series = [series[i] for i in idx]
y = y[idx]

# split
split = int(0.95 * len(series))
train_series = series[:split]
train_labels = y[:split]
test_series = series[split:]
test_labels = y[split:]

ds_train, ds_test = TimeSeriesDataset(train_series), TimeSeriesDataset(test_series)

collate_fn = functools.partial(
    contrastive_collate,
    min_len = COLLATE_FN_KWARGS['min_len'],
    max_len = COLLATE_FN_KWARGS['max_len'],
    num_neg = COLLATE_FN_KWARGS['num_neg'],
    min_overlap = COLLATE_FN_KWARGS['min_overlap'],
    min_var = COLLATE_FN_KWARGS['min_var'],
    max_var = COLLATE_FN_KWARGS['max_var']
)

dl_train = DataLoader(
    ds_train,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collate_fn,
    num_workers = N_WORKERS,
    pin_memory = True,
    drop_last = True
)

dl_test = DataLoader(
    ds_test,
    batch_size = len(test_series),
    shuffle = False,
    num_workers = N_WORKERS,
    pin_memory = True,
)

In [5]:
### TRAINING

for i, dim in enumerate(REP_DIMS):
    print(f'TRAINING MODEL WITH REPRESENTATION DIMENSION {dim}')
    history = run_training(
        model = MODELS[i],
        train_loader = dl_train,
        val_loader = None,
        device = DEVICE,
        optimizer_dict = OPTIMIZER_DICTS[i],
        epochs_stage1 = N_EPOCHS_STAGE1,
        epochs_stage2 = N_EPOCHS_STAGE2,
        epochs_stage3 = N_EPOCHS_STAGE3,
        scheduler_dict = SCHEDULER,
        mid_weight = MID_WEIGHT,
        proto_neg_margin = PROTO_NEG_MARGIN,
        proto_diversity_threshold = PROTO_DIVERSITY_THRESHOLD,
        lambda_proto = LAMBDA_PROTO,
        temperature = TEMPERATURE,
        lambda_repr = LAMBDA_REPR,
        grad_clip_norm = GRADIENT_CLIP,
        checkpoint_path = CKPT_PATHS[i],
        checkpoint_epochs = CHECKPOINT_EPOCHS,
        collector_fn = None,
        use_amp = True
    )

TRAINING MODEL WITH REPRESENTATION DIMENSION 300
Training for 55 epochs.

=== stage1 (30 epochs) ===
[stage1] epoch 1/30 | global 1/55
  train loss: 3.651870
Saved checkpoint at epoch 1
[stage1] epoch 2/30 | global 2/55
  train loss: 1.776059
Saved checkpoint at epoch 2
[stage1] epoch 3/30 | global 3/55
  train loss: 1.498382
Saved checkpoint at epoch 3
[stage1] epoch 4/30 | global 4/55
  train loss: 1.477324
Saved checkpoint at epoch 4
[stage1] epoch 5/30 | global 5/55
  train loss: 1.465904
Saved checkpoint at epoch 5
[stage1] epoch 6/30 | global 6/55
  train loss: 1.460186
Saved checkpoint at epoch 6
[stage1] epoch 7/30 | global 7/55
  train loss: 1.445884
Saved checkpoint at epoch 7
[stage1] epoch 8/30 | global 8/55
  train loss: 1.444475
Saved checkpoint at epoch 8
[stage1] epoch 9/30 | global 9/55
  train loss: 1.435979
Saved checkpoint at epoch 9
[stage1] epoch 10/30 | global 10/55
  train loss: 1.430827
Saved checkpoint at epoch 10
[stage1] epoch 11/30 | global 11/55
  train lo

In [ ]:
runtime.unassign()